# 02 — Data Preparation

This notebook consumes the persisted exploration handoff from Notebook 01, independently reacquires and validates the UCI Dry Bean source, creates an unchanged defensive prepared projection, freezes a stratified multiclass split, and publishes a reloadable handoff for Notebook 03. It performs no model fitting, scoring, threshold selection, feature selection, resampling, or final-test evaluation.

## 1. Preparation Context and Handoff Boundary

Notebook 02 starts from a fresh kernel. The persisted Notebook-01 handoff is authoritative for scientific roles and split policy; live variables from exploration are neither required nor accepted. The released table is treated as a static educational classification snapshot, while operational validity and inference-time feature availability remain unconfirmed.

In [3]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from scripts.build_exploration_handoff import load_and_validate_exploration_handoff
from scripts.download_data import acquire_uci_dataset
from scripts.prepare_data import (
    ClassificationSplitPolicy,
    analyze_repeated_profiles_across_partitions,
    build_feature_manifest,
    build_preparation_handoff_manifest,
    build_preparation_manifest,
    build_quality_evidence,
    build_split_manifest,
    fingerprint_dataframe,
    fingerprint_dataframe_csv,
    fingerprint_file,
    load_and_validate_preparation_handoff,
    prepare_tabular_dataset,
    separate_dataset_roles,
    split_classification_dataset,
    validate_dataset_partitions,
    validate_prepared_dataset,
    validate_raw_dataset,
    validate_source_against_exploration_handoff,
    write_preparation_artifacts,
)
from scripts.project_context import get_project_context

PROJECT = get_project_context()
DATASET_SLUG = "dry-bean"
UCI_DATASET_ID = 602
SOURCE_REPOSITORY = "UCI Machine Learning Repository"
EXPLORATION_HANDOFF_PATH = Path("artifacts/exploration/dry-bean/exploration-handoff.json")

print(f"Project: {PROJECT.name}")
print(f"Required exploration handoff: {EXPLORATION_HANDOFF_PATH.as_posix()}")

Project: dataset-study-dry-bean
Required exploration handoff: artifacts/exploration/dry-bean/exploration-handoff.json


## 2. Independent Exploration-Handoff Loading

The loader validates the portable Notebook-01 schema and readiness gates before any source acquisition or preparation occurs. Dataset-specific roles below are reconstructed from that artifact rather than restated as independent policy.

In [4]:
exploration_handoff = load_and_validate_exploration_handoff(
    PROJECT.require_file(EXPLORATION_HANDOFF_PATH),
    expected_dataset_slug=DATASET_SLUG,
    expected_source_dataset_id=UCI_DATASET_ID,
)

source_contract = exploration_handoff["source"]
prediction_contract = exploration_handoff["prediction_contract"]
feature_contract = exploration_handoff["feature_contract"]
split_contract = exploration_handoff["preparation_contract"]["split_policy"]

TARGET_COLUMN = prediction_contract["target_column"]
TARGET_CLASSES = tuple(prediction_contract["target_classes"])
FEATURE_COLUMNS = tuple(feature_contract["feature_columns"])
NUMERICAL_FEATURES = tuple(feature_contract["numerical_features"])
CATEGORICAL_FEATURES = tuple(feature_contract["categorical_features"])
IDENTIFIER_COLUMNS = tuple(feature_contract["identifier_columns"])

assert prediction_contract["problem_type"] == "multiclass_classification"
assert prediction_contract["positive_class"] is None
assert prediction_contract["class_semantics"] == "Nominal / unordered"
assert FEATURE_COLUMNS == NUMERICAL_FEATURES
assert CATEGORICAL_FEATURES == ()
assert IDENTIFIER_COLUMNS == ()
assert exploration_handoff["readiness"]["deterministic_preparation_ready"] is True
assert exploration_handoff["readiness"]["split_execution_ready"] is True

display(pd.DataFrame([{
    "problem_type": prediction_contract["problem_type"],
    "target": TARGET_COLUMN,
    "classes": len(TARGET_CLASSES),
    "features": len(FEATURE_COLUMNS),
    "source_identifiers": len(IDENTIFIER_COLUMNS),
    "notebook_01_handoff_validated": True,
}]))

,problem_type,target,classes,features,source_identifiers,notebook_01_handoff_validated
0,multiclass_classification,Class,7,16,0,True


## 3. Independent UCI Source Acquisition

The existing UCI integration reuses a complete local materialization or calls `fetch_ucirepo(id=602)` when it is absent. `dataset.csv`, `metadata.json`, and `variables.csv` are all required; no Notebook-01 DataFrame is reused.

In [5]:
acquisition = acquire_uci_dataset(
    dataset_id=int(source_contract["dataset_id"]),
    destination=Path("data") / "raw" / DATASET_SLUG,
    project_root=PROJECT.root,
)
DATASET_FILE = acquisition.require_one_file("dataset.csv")
METADATA_FILE = acquisition.require_one_file("metadata.json")
VARIABLES_FILE = acquisition.require_one_file("variables.csv")
raw_df = pd.read_csv(DATASET_FILE)

print(f"Source: {acquisition.source_reference}")
print(f"Dataset file: {PROJECT.display(DATASET_FILE)}")
print(f"Loaded shape: {raw_df.shape}")

Source: UCI ML Repository dataset 602
Dataset file: data/raw/dry-bean/dataset.csv
Loaded shape: (13611, 17)


## 4. Source Identity and Contract Revalidation

This fail-closed gate compares source bytes, logical path, shape, column order, target, class set, feature order, identifier absence, problem type, UCI metadata ID, and UCI variable roles with the exploration handoff. Any divergence stops preparation explicitly.

In [6]:
source_identity = validate_source_against_exploration_handoff(
    raw_df,
    handoff=exploration_handoff,
    source_file=DATASET_FILE,
    project_root=PROJECT.root,
    dataset_slug=DATASET_SLUG,
    source_repository=SOURCE_REPOSITORY,
    source_dataset_id=UCI_DATASET_ID,
    metadata_file=METADATA_FILE,
    variables_file=VARIABLES_FILE,
)

COLUMN_ORDER = source_identity.column_order
EXPECTED_DTYPES = {
    **{feature: "numeric" for feature in FEATURE_COLUMNS},
    TARGET_COLUMN: "string",
}
raw_validation = validate_raw_dataset(
    raw_df,
    column_order=COLUMN_ORDER,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    categorical_expected_values={},
    expected_types=EXPECTED_DTYPES,
)

RAW_SNAPSHOT = raw_df.copy(deep=True)
RAW_LOGICAL_FINGERPRINT = fingerprint_dataframe(raw_df)
SOURCE_SHA256 = fingerprint_file(DATASET_FILE)
display(pd.DataFrame([source_identity.as_dict()]).drop(columns=["checks"]))
print("Source identity gate: PASSED")

,dataset_slug,source_repository,source_dataset_id,source_path,source_sha256,row_count,column_count,column_order,target_column,target_classes,feature_columns,feature_count,identifier_columns,identifier_count,problem_type,valid
0,dry-bean,UCI Machine Learning Repository,602,data/raw/dry-bean/dataset.csv,1330e4ccc5c54a925e43daf60d1409ac62dad2a21de9a2...,13611,17,"[Area, Perimeter, MajorAxisLength, MinorAxisLe...",Class,"[SEKER, BARBUNYA, BOMBAY, CALI, DERMASON, HORO...","[Area, Perimeter, MajorAxisLength, MinorAxisLe...",16,[],0,multiclass_classification,True


Source identity gate: PASSED


## 5. Defensive Prepared Projection

Notebook 01 authorized no deterministic value repair. Preparation is therefore a deep defensive copy with zero materialization rules: no rows, columns, values, labels, or source bytes may change.

In [7]:
prepared_result = prepare_tabular_dataset(raw_df)
prepared_df = prepared_result.dataframe

prepared_validation = validate_prepared_dataset(
    raw_df,
    prepared_df,
    column_order=COLUMN_ORDER,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    categorical_expected_values={},
    expected_types=EXPECTED_DTYPES,
    authorized_changed_columns=(),
    expected_row_count=source_identity.row_count,
    expected_materialized_counts={},
    observed_materialized_counts=dict(prepared_result.materialized_counts),
)

pd.testing.assert_frame_equal(raw_df, RAW_SNAPSHOT)
pd.testing.assert_frame_equal(prepared_df, RAW_SNAPSHOT)
assert prepared_df is not raw_df
assert fingerprint_dataframe(raw_df) == RAW_LOGICAL_FINGERPRINT
assert fingerprint_file(DATASET_FILE) == SOURCE_SHA256
assert prepared_result.rules == ()

print("Rows removed: 0")
print("Values changed: 0")
print("Materialization rules applied: 0")
print("Raw source unchanged: True")

Rows removed: 0
Values changed: 0
Materialization rules applied: 0
Raw source unchanged: True


## 6. Feature and Target Separation

The predictor projection contains the 16 ordered numerical features. Lineage is intentionally empty because the source provides no observation identifier. `Class` remains readable and nominal; the deterministic class order is a technical contract, not an ordinal ranking.

In [8]:
roles = separate_dataset_roles(
    prepared_df,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
)
lineage = roles.lineage
X = roles.features
y = roles.target
TARGET_ENCODING_CONTRACT = {
    target_class: index
    for index, target_class in enumerate(TARGET_CLASSES)
}

assert lineage.shape == (len(prepared_df), 0)
assert tuple(X.columns) == FEATURE_COLUMNS
assert TARGET_COLUMN not in X.columns
assert set(y) == set(TARGET_CLASSES)

display(pd.DataFrame([{
    "numerical_features": len(NUMERICAL_FEATURES),
    "categorical_features": len(CATEGORICAL_FEATURES),
    "source_identifiers": len(IDENTIFIER_COLUMNS),
    "target_classes": len(TARGET_CLASSES),
    "persisted_target_labels_readable": True,
    "positive_class": None,
}]))

,numerical_features,categorical_features,source_identifiers,target_classes,persisted_target_labels_readable,positive_class
0,16,0,0,7,True,None


## 7. Split Policy Reconstruction

Fractions, stratification field, and seed come directly from the exploration handoff. `resolved_static_snapshot` records that the released table has no chronological evaluation field; it does not claim temporal, future, or production validity.

In [9]:
SPLIT_POLICY = ClassificationSplitPolicy(
    evaluation_mode="stratified_random_snapshot",
    purpose="educational_benchmark",
    train_fraction=float(split_contract["train_fraction"]),
    validation_fraction=float(split_contract["validation_fraction"]),
    test_fraction=float(split_contract["test_fraction"]),
    stratify_by=split_contract["stratification_field"],
    random_seed=int(split_contract["random_seed"]),
    shuffle=True,
    educational_justification=(
        "Use the source-released static Dry Bean snapshot for a reproducible "
        "educational multiclass benchmark without claiming future or "
        "production validity."
    ),
    operational_validity="unconfirmed",
    temporal_contract_status="resolved_static_snapshot",
    feature_inference_availability="unconfirmed",
)

assert split_contract["temporal_policy_status"] == "Resolved snapshot fallback"
assert split_contract["test_holdout_untouched"] is True
assert split_contract["disjoint_partitions_required"] is True
assert tuple(split_contract["identifier_grouping"]) == IDENTIFIER_COLUMNS
display(pd.DataFrame([SPLIT_POLICY.as_dict()]).drop(columns=["stage_seeds"]))
print("Stage seeds:", SPLIT_POLICY.as_dict()["stage_seeds"] )

,evaluation_mode,purpose,train_fraction,validation_fraction,test_fraction,stratify_by,random_seed,shuffle,educational_justification,operational_validity,temporal_contract_status,feature_inference_availability
0,stratified_random_snapshot,educational_benchmark,0.7,0.15,0.15,Class,42,True,Use the source-released static Dry Bean snapsh...,unconfirmed,resolved_static_snapshot,unconfirmed


Stage seeds: {'train_vs_temporary': 42, 'validation_vs_test': 43}


## 8. Stratified Multiclass Partitioning

Without a source identifier, deterministic membership uses a technical full-row hash plus occurrence ordinal. This representation is persisted only as integrity evidence: it is not added to the CSV, is never a predictor, and does not assert real-world grain identity.

In [10]:
partitions = split_classification_dataset(
    prepared_df,
    policy=SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=TARGET_CLASSES,
)
train_df = partitions.train
validation_df = partitions.validation
test_df = partitions.test

display(pd.DataFrame([
    {"partition": name, "rows": len(frame)}
    for name, frame in partitions.as_mapping().items()
]))
print("Split method:", partitions.split_method)
print("Membership kind:", partitions.membership_kind)

,partition,rows
0,train,9527
1,validation,2042
2,test,2042


Split method: two_stage_sklearn_train_test_split_with_stratification_and_technical_row_occurrence_sorted_membership
Membership kind: technical_row_occurrence


## 9. Partition Integrity and Repeated-Profile Review

Validation proves row-multiset coverage, isolation of technical occurrence membership, class presence, stratification tolerance, stable source order, and same-policy reproducibility. Exact equality and repeated feature profiles are reported without being relabeled as proven same-grain leakage.

In [11]:
PARTITION_PREVALENCE_TOLERANCE = 0.01
partition_validation = validate_dataset_partitions(
    prepared_df,
    partitions,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    prevalence_tolerance=PARTITION_PREVALENCE_TOLERANCE,
)
repeat_partitions = split_classification_dataset(
    prepared_df,
    policy=SPLIT_POLICY,
    identifier_columns=IDENTIFIER_COLUMNS,
    target_classes=TARGET_CLASSES,
)
assert partitions.membership_mapping() == repeat_partitions.membership_mapping()

repeated_profile_evidence = analyze_repeated_profiles_across_partitions(
    prepared_df,
    partitions,
    feature_columns=FEATURE_COLUMNS,
    target_column=TARGET_COLUMN,
    identifier_columns=IDENTIFIER_COLUMNS,
)

class_rows = []
for partition_name, frame in partitions.as_mapping().items():
    counts = frame[TARGET_COLUMN].value_counts().reindex(TARGET_CLASSES, fill_value=0)
    class_rows.extend({
        "partition": partition_name,
        "class": target_class,
        "count": int(counts[target_class]),
        "proportion": float(counts[target_class] / len(frame)),
    } for target_class in TARGET_CLASSES)
display(pd.DataFrame(class_rows))
display(pd.DataFrame([repeated_profile_evidence]).drop(columns=[
    "cross_partition_exact_row_samples",
    "cross_partition_feature_profile_samples",
]))
print("Partition integrity valid:", partition_validation.is_valid)
print("Entity disjointness status:", partition_validation.entity_disjointness_status)

,partition,class,count,proportion
0,train,SEKER,1419,0.148945
1,train,BARBUNYA,925,0.097092
2,train,BOMBAY,365,0.038312
3,train,CALI,1141,0.119765
4,train,DERMASON,2482,0.260523
5,train,HOROZ,1350,0.141703
6,train,SIRA,1845,0.193660
7,validation,SEKER,304,0.148874
8,validation,BARBUNYA,198,0.096964
9,validation,BOMBAY,79,0.038688


,evidence_type,source_identifier_available,identity_interpretation,proven_duplicate_identity,source_exact_row_equality_group_count,source_exact_row_equality_row_count,source_repeated_feature_profile_group_count,cross_partition_exact_row_equality_group_count,cross_partition_repeated_feature_profile_group_count,target_conflicting_feature_profile_group_count,exact_row_multiplicity_preserved,feature_profile_multiplicity_preserved
0,repeated_profile_partition_review,False,row equality is observational evidence only; d...,False,68,136,68,31,31,0,True,True


Partition integrity valid: True
Entity disjointness status: not_claimed_without_source_identifiers


## 10. Model-Selection Preprocessing Contract

Notebook 02 fits no learned preprocessing. Notebook 03 may fit model-specific scaling, selection, weighting, resampling, or other learned transformations only inside training data or training folds. Validation may compare frozen candidates; test remains structurally sealed for the later final-evaluation stage.

In [12]:
PREPROCESSING_CONTRACT = {
    "input_feature_type": "numerical_only",
    "categorical_strategy": "not_applicable",
    "numerical_scaling": "model_specific",
    "learned_fit_scope": "inside_training_data_or_training_fold_only",
    "learned_transformations_fitted_in_notebook_02": False,
    "persisted_target": "readable_nominal_labels",
    "target_class_order_semantics": "deterministic_not_ordinal",
    "deferred_to_notebook_03": [
        "multiclass Dummy baseline comparison",
        "cross-validation",
        "model and hyperparameter selection",
        "model-specific scaling",
        "class weighting and training-only resampling",
        "SMOTE or equivalent only if later justified",
        "feature selection and redundancy ablation",
        "ShapeFactor2-specific analysis",
        "overlap-hypothesis evaluation",
        "confusion matrices and macro/micro/weighted/per-class metrics",
        "calibration analysis if appropriate",
    ],
    "prohibited_in_notebook_02": [
        "model fit",
        "model score",
        "candidate selection",
        "threshold tuning",
        "test-set decision use",
    ],
    "test_partition_sealed": True,
}

display(pd.DataFrame([{
    "input_feature_type": PREPROCESSING_CONTRACT["input_feature_type"],
    "categorical_strategy": PREPROCESSING_CONTRACT["categorical_strategy"],
    "learned_fit_scope": PREPROCESSING_CONTRACT["learned_fit_scope"],
    "learned_transformations_fitted": False,
    "test_partition_sealed": True,
}]))

,input_feature_type,categorical_strategy,learned_fit_scope,learned_transformations_fitted,test_partition_sealed
0,numerical_only,not_applicable,inside_training_data_or_training_fold_only,False,True


## 11. Preparation Artifact Contracts

The prepared CSV and frozen partitions use deterministic serialization. The four required manifests capture source identity, feature/target roles, split membership, quality evidence, preservation, and readiness. A compact `preparation-handoff.json` authenticates those four JSON components by path and SHA-256 so Notebook 03 can enter through one integrity gate.

In [13]:
SPLIT_ID = (
    f"stratified-{int(SPLIT_POLICY.train_fraction * 100)}-"
    f"{int(SPLIT_POLICY.validation_fraction * 100)}-"
    f"{int(SPLIT_POLICY.test_fraction * 100)}-seed-{SPLIT_POLICY.random_seed}"
)
PREPARED_PATH = Path("data/processed") / DATASET_SLUG / "prepared.csv"
SPLIT_ROOT = Path("data/processed") / DATASET_SLUG / "splits" / SPLIT_ID
PARTITION_PATHS = {
    name: SPLIT_ROOT / f"{name}.csv"
    for name in ("train", "validation", "test")
}
ARTIFACT_ROOT = Path("artifacts/preparation") / DATASET_SLUG
MANIFEST_PATHS = {
    "preparation_manifest": ARTIFACT_ROOT / "preparation-manifest.json",
    "feature_manifest": ARTIFACT_ROOT / "feature-manifest.json",
    "split_manifest": ARTIFACT_ROOT / "split-manifest.json",
    "quality_evidence": ARTIFACT_ROOT / "quality-evidence.json",
}
PREPARATION_HANDOFF_PATH = ARTIFACT_ROOT / "preparation-handoff.json"

PREPARED_SHA256 = fingerprint_dataframe_csv(prepared_df)
PARTITION_SHA256 = {
    name: fingerprint_dataframe_csv(frame)
    for name, frame in partitions.as_mapping().items()
}
READINESS = {
    "notebook_01_handoff_validated": True,
    "source_independently_revalidated": True,
    "prepared_projection_materialized": True,
    "split_materialized": True,
    "partition_integrity_validated": True,
    "preparation_handoff_reloadable": True,
    "educational_model_selection_ready": True,
    "test_partition_sealed": True,
    "test_partition_evaluated": False,
    "model_selected": False,
    "final_model_trained": False,
    "operational_modeling_ready": False,
    "operational_validity": "unconfirmed",
    "temporal_contract_status": "resolved_static_snapshot",
    "feature_inference_availability": "unconfirmed",
}

In [14]:
preparation_manifest = build_preparation_manifest(
    dataset_slug=DATASET_SLUG,
    source_path=source_identity.source_path,
    source_sha256=SOURCE_SHA256,
    prepared_path=PREPARED_PATH,
    prepared_sha256=PREPARED_SHA256,
    raw_report=raw_validation,
    prepared_report=prepared_validation,
    preparation=prepared_result,
    raw_fingerprint_before=RAW_LOGICAL_FINGERPRINT,
    raw_fingerprint_after=fingerprint_dataframe(raw_df),
    source_sha256_after=fingerprint_file(DATASET_FILE),
    deterministic_rules=[],
    readiness=READINESS,
    source_identity=source_identity.as_dict(),
)
feature_manifest = build_feature_manifest(
    dataset_slug=DATASET_SLUG,
    identifier_columns=IDENTIFIER_COLUMNS,
    feature_columns=FEATURE_COLUMNS,
    numerical_features=NUMERICAL_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
    categorical_expected_values={},
    target_column=TARGET_COLUMN,
    target_classes=TARGET_CLASSES,
    expected_dtypes={"raw": EXPECTED_DTYPES, "prepared": EXPECTED_DTYPES},
    preprocessing_contract=PREPROCESSING_CONTRACT,
    prohibited_predictors=(TARGET_COLUMN, "target-derived values", "technical membership tokens"),
    positive_target_class=None,
    target_encoding=TARGET_ENCODING_CONTRACT,
    problem_type=prediction_contract["problem_type"],
    target_semantics="nominal_unordered",
)
split_manifest = build_split_manifest(
    dataset_slug=DATASET_SLUG,
    policy=SPLIT_POLICY,
    partitions=partitions,
    validation=partition_validation,
    partition_paths=PARTITION_PATHS,
    partition_sha256=PARTITION_SHA256,
    repeated_profile_evidence=repeated_profile_evidence,
)
quality_evidence = build_quality_evidence(
    dataset_slug=DATASET_SLUG,
    raw_report=raw_validation,
    prepared_report=prepared_validation,
    partition_report=partition_validation,
    preparation=prepared_result,
    fingerprints={
        "source_sha256_before": SOURCE_SHA256,
        "source_sha256_after": fingerprint_file(DATASET_FILE),
        "raw_logical_fingerprint_before": RAW_LOGICAL_FINGERPRINT,
        "raw_logical_fingerprint_after": fingerprint_dataframe(raw_df),
        "prepared_sha256": PREPARED_SHA256,
        "partition_sha256": PARTITION_SHA256,
    },
    readiness=READINESS,
    preservation_checks={
        "raw_dataframe_unchanged": True,
        "source_file_unchanged": True,
        "rows_removed": 0,
        "values_changed": 0,
        "features_removed": 0,
        "target_classes_removed": 0,
        "duplicate_rows_removed": 0,
        "generic_outlier_treatment_applied": False,
        "learned_preprocessing_fitted": False,
    },
    repeated_profile_evidence=repeated_profile_evidence,
)

COMPONENT_PAYLOADS = {
    "preparation_manifest": preparation_manifest,
    "feature_manifest": feature_manifest,
    "split_manifest": split_manifest,
    "quality_evidence": quality_evidence,
}
preparation_handoff_manifest = build_preparation_handoff_manifest(
    dataset_slug=DATASET_SLUG,
    component_paths=MANIFEST_PATHS,
    component_payloads=COMPONENT_PAYLOADS,
    readiness=READINESS,
)

## 12. Atomic Artifact Materialization

All CSV and JSON outputs are staged and validated as one set. Equivalent reruns are accepted; divergent runtime artifacts fail closed unless a caller explicitly authorizes overwrite. This notebook never authorizes overwrite.

In [15]:
write_result = write_preparation_artifacts(
    project_root=PROJECT.root,
    csv_artifacts={
        PREPARED_PATH: prepared_df,
        **{
            PARTITION_PATHS[name]: frame
            for name, frame in partitions.as_mapping().items()
        },
    },
    json_artifacts={
        **{
            MANIFEST_PATHS[name]: payload
            for name, payload in COMPONENT_PAYLOADS.items()
        },
        PREPARATION_HANDOFF_PATH: preparation_handoff_manifest,
    },
    overwrite=False,
)

display(pd.DataFrame([
    {"path": path, "status": status, "sha256": dict(write_result.sha256)[path]}
    for path, status in write_result.statuses
]))

,path,status,sha256
0,artifacts/preparation/dry-bean/feature-manifes...,created,c12b3d2ec9d7efdc65d922a966e9e4390529579618b71d...
1,artifacts/preparation/dry-bean/preparation-han...,created,ad6673e3c5ba5e1d91a1b372442b9a484d704cc7f56bef...
2,artifacts/preparation/dry-bean/preparation-man...,created,bf7889c9909ec25ac36b809ce4ff365d66ab7b425f1cdc...
3,artifacts/preparation/dry-bean/quality-evidenc...,created,5bceca6bbf8503747e227ee569d35d12d171987e8bc48d...
4,artifacts/preparation/dry-bean/split-manifest....,created,6713f56139eff8035f04088bfa8b23535e2e55e406f844...
5,data/processed/dry-bean/prepared.csv,created,9b6db864219c22a9255e2fad23a8267ea3cdb3dbe252c3...
6,data/processed/dry-bean/splits/stratified-70-1...,created,5e17c1346e67cb4a786ebd11162b7e45099ded342b9de9...
7,data/processed/dry-bean/splits/stratified-70-1...,created,485594b308ad5e6bb36bfe6a7758ded01c061ed5062995...
8,data/processed/dry-bean/splits/stratified-70-1...,created,f379db8842c3de49d70c7fe4dd449cdc3220052f9d7bdd...


## 13. Preparation Findings and Readiness

Preparation is complete only at the data-contract boundary. Educational model selection is ready; no candidate, model, threshold, final metric, or production claim exists.

In [16]:
display(pd.DataFrame([{
    "notebook_01_handoff_validated": True,
    "source_independently_revalidated": True,
    "prepared_projection_materialized": True,
    "split_materialized": True,
    "partition_integrity_validated": True,
    "educational_model_selection_ready": True,
    "test_partition_sealed": True,
    "test_partition_evaluated": False,
    "model_selected": False,
    "final_model_trained": False,
    "operational_modeling_ready": False,
}]))

,notebook_01_handoff_validated,source_independently_revalidated,prepared_projection_materialized,split_materialized,partition_integrity_validated,educational_model_selection_ready,test_partition_sealed,test_partition_evaluated,model_selected,final_model_trained,operational_modeling_ready
0,True,True,True,True,True,True,True,False,False,False,False


## 14. Model-Selection Handoff Validation

A real reload now discards reliance on the live construction objects. The composite handoff authenticates the four manifest files; the loader then reopens prepared/train/validation/test CSVs, verifies every hash, validates technical membership and row multiplicity after `pd.read_csv`, and recovers the frozen feature, target, split, preprocessing, quality, and readiness contracts.

In [17]:
reloaded_handoff = load_and_validate_preparation_handoff(
    project_root=PROJECT.root,
    preparation_handoff_path=PREPARATION_HANDOFF_PATH,
)
reloaded_manifests = reloaded_handoff.manifests
reloaded_feature_manifest = reloaded_manifests["feature_manifest"]
reloaded_split_manifest = reloaded_manifests["split_manifest"]
reloaded_quality_evidence = reloaded_manifests["quality_evidence"]

assert len(reloaded_handoff.prepared) == source_identity.row_count
assert tuple(reloaded_handoff.prepared.columns) == COLUMN_ORDER
assert tuple(reloaded_feature_manifest["feature_columns"]) == FEATURE_COLUMNS
assert tuple(reloaded_feature_manifest["target_classes"]) == TARGET_CLASSES
assert reloaded_feature_manifest["problem_type"] == "multiclass_classification"
assert reloaded_feature_manifest["positive_target_class"] is None
assert reloaded_feature_manifest["target_contract"]["semantics"] == "nominal_unordered"
assert reloaded_split_manifest["membership_kind"] == "technical_row_occurrence"
assert reloaded_split_manifest["entity_disjointness_status"] == "not_claimed_without_source_identifiers"
assert reloaded_quality_evidence["readiness"]["educational_model_selection_ready"] is True
assert reloaded_quality_evidence["readiness"]["test_partition_sealed"] is True
assert reloaded_quality_evidence["readiness"]["test_partition_evaluated"] is False
assert reloaded_quality_evidence["readiness"]["model_selected"] is False
assert reloaded_quality_evidence["readiness"]["final_model_trained"] is False

display(pd.DataFrame([{
    "handoff_path": PREPARATION_HANDOFF_PATH.as_posix(),
    "prepared_rows": len(reloaded_handoff.prepared),
    "train_rows": len(reloaded_handoff.train),
    "validation_rows": len(reloaded_handoff.validation),
    "test_rows_structural_only": len(reloaded_handoff.test),
    "features": len(reloaded_feature_manifest["feature_columns"]),
    "classes": len(reloaded_feature_manifest["target_classes"]),
    "test_metrics_calculated": False,
    "model_selection_ready": True,
}]))
print("Preparation handoff reload: PASSED")
print("Notebook 03 may begin model selection from the frozen handoff; test evaluation remains prohibited.")

,handoff_path,prepared_rows,train_rows,validation_rows,test_rows_structural_only,features,classes,test_metrics_calculated,model_selection_ready
0,artifacts/preparation/dry-bean/preparation-han...,13611,9527,2042,2042,16,7,False,True


Preparation handoff reload: PASSED
Notebook 03 may begin model selection from the frozen handoff; test evaluation remains prohibited.
